# Temporal ERA5 -> PRISM Downscaling (Georgia)

This notebook demonstrates a compact temporal baseline for climate downscaling:
- input: ERA5 sequence `[t-k+1 ... t]`
- target: PRISM map at time `t`
- model: CNN that uses time as channels


## 1) ERA5 vs PRISM
ERA5 is coarse but temporally rich. PRISM is finer resolution and used here as the learning target.
The task is temporal downscaling: use multiple recent ERA5 steps to predict one PRISM map.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datasets.prism_dataset import ERA5_PRISM_Dataset
from models.cnn_downscaler import CNNDownscaler


In [ ]:
ERA5_PATH = ROOT / 'data_raw' / 'era5_georgia_temp.nc'
PRISM_PATH = ROOT / 'data_raw' / 'prism'
CHECKPOINT_PATH = ROOT / 'checkpoints' / 'cnn_downscaler_best.pt'
RESULTS_DIR = ROOT / 'results' / 'evaluation'
HISTORY_LENGTH = 3

print('ERA5_PATH:', ERA5_PATH)
print('PRISM_PATH:', PRISM_PATH)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('RESULTS_DIR:', RESULTS_DIR)
print('HISTORY_LENGTH:', HISTORY_LENGTH)


## 2) Temporal Dataset
Each sample is built as:
- `X`: ERA5 window `[t-k+1 ... t]` with shape `(T, H, W)`
- `Y`: PRISM target at `t` with shape `(1, H_high, W_high)`


In [ ]:
if not ERA5_PATH.exists():
    raise FileNotFoundError(f'Missing ERA5 file: {ERA5_PATH}')
if not PRISM_PATH.exists():
    raise FileNotFoundError(f'Missing PRISM directory: {PRISM_PATH}')

dataset = ERA5_PRISM_Dataset(
    era5_path=str(ERA5_PATH),
    prism_path=str(PRISM_PATH),
    history_length=HISTORY_LENGTH,
)
x0, y0 = dataset[0]
meta0 = dataset.metadata(0)
print('Aligned samples:', len(dataset))
print('Sample date:', meta0.date.date())
print('X shape:', tuple(x0.shape))
print('Y shape:', tuple(y0.shape))


## 3) Model Overview
The CNN treats temporal dimension `T` as channels.
This gives a simple temporal baseline before moving to ConvLSTM or transformer encoders.


In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_config = checkpoint.get('model_config', {}) if isinstance(checkpoint, dict) else {}
model = CNNDownscaler(
    in_channels=int(model_config.get('in_channels', HISTORY_LENGTH)),
    out_channels=int(model_config.get('out_channels', 1)),
    base_channels=int(model_config.get('base_channels', 32)),
)
model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
model = model.to(device).eval()
print('Loaded model on', device)


## 4) Inference and Pipeline Output
Run one sample through the trained model and compute RMSE/MAE.


In [ ]:
date = dataset.metadata(0).date
x, y = dataset[0]
x_batch = x.unsqueeze(0).to(device)
y_batch = y.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x_batch, target_size=(y.shape[-2], y.shape[-1]))

rmse = torch.sqrt(torch.mean((pred - y_batch) ** 2)).item()
mae = torch.mean(torch.abs(pred - y_batch)).item()
print('Date:', date.date())
print(f'Sample RMSE: {rmse:.4f}')
print(f'Sample MAE : {mae:.4f}')


## 5) Visualization: Input vs Prediction vs Ground Truth
For temporal input, the left panel shows the most recent ERA5 frame (`t`).


In [ ]:
era5_t = x[-1].unsqueeze(0).unsqueeze(0)
era5_up = F.interpolate(
    era5_t,
    size=(y.shape[-2], y.shape[-1]),
    mode='bilinear',
    align_corners=False,
).squeeze().cpu().numpy()

pred_map = pred.squeeze().cpu().numpy()
target_map = y.squeeze().cpu().numpy()

vmin = min(np.min(era5_up), np.min(pred_map), np.min(target_map))
vmax = max(np.max(era5_up), np.max(pred_map), np.max(target_map))

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
im = axes[0].imshow(era5_up, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0].set_title('ERA5 t (Upsampled)')
axes[0].axis('off')

axes[1].imshow(pred_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1].set_title('CNN Prediction')
axes[1].axis('off')

axes[2].imshow(target_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[2].set_title('PRISM Target')
axes[2].axis('off')

fig.suptitle(f'Temporal Downscaling | {date.date()} | history={HISTORY_LENGTH}')
fig.colorbar(im, ax=axes, shrink=0.8, label='Temperature (deg C)')
plt.show()


## Saved Evaluation Results
Load outputs from `results/evaluation/` if they exist.


In [ ]:
saved_plot = RESULTS_DIR / 'comparison.png'
saved_metrics = RESULTS_DIR / 'metrics.json'

if saved_plot.exists():
    display(Markdown('### Saved comparison.png'))
    display(Image(filename=str(saved_plot)))
else:
    cmd = ('python evaluation/evaluate_model.py --history-length 3 '           '--checkpoint-path checkpoints/cnn_downscaler_best.pt '           '--num-samples 8 --num-plots 1 --results-dir results/evaluation')
    display(Markdown(f'`comparison.png` not found. Run:\n\n`{cmd}`'))

if saved_metrics.exists():
    metrics = json.loads(saved_metrics.read_text())
    display(Markdown(f"Metrics: RMSE={metrics['rmse']:.4f}, MAE={metrics['mae']:.4f}, history_length={metrics.get('history_length', 'n/a')}"))
